In [1]:
import cv2
import torch
import numpy as np
from pathlib import Path
from deep_sort_realtime.deepsort_tracker import DeepSort
from collections import deque
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array
from PIL import Image
from transformers import pipeline
import datetime
from typing import Dict
import os
import json
from langchain.llms import Ollama
from tensorflow.keras.models import load_model


## OBJECT DETECTION (PERSON)
#### YOLOV5

In [2]:
import warnings
import torch 
import sys
import os

# Add the yolov5 directory to path
sys.path.append(r"C:\Users\ASUS\Women Safety Ecosystem\yolov5\models\yolov5s.yaml")  

warnings.simplefilter(action='ignore', category=FutureWarning)

# Load YOLOv5 model
model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)
model.classes = [0]  # Only detect persons (class 0 in COCO dataset)
model.conf = 0.5  # Confidence threshold
model.cpu()  # Ensure model runs on CPU

Using cache found in C:\Users\ASUS/.cache\torch\hub\ultralytics_yolov5_master
YOLOv5  2024-12-4 Python-3.12.1 torch-2.5.1+cpu CPU

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


AutoShape(
  (model): DetectMultiBackend(
    (model): DetectionModel(
      (model): Sequential(
        (0): Conv(
          (conv): Conv2d(3, 32, kernel_size=(6, 6), stride=(2, 2), padding=(2, 2))
          (act): SiLU(inplace=True)
        )
        (1): Conv(
          (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (act): SiLU(inplace=True)
        )
        (2): C3(
          (cv1): Conv(
            (conv): Conv2d(64, 32, kernel_size=(1, 1), stride=(1, 1))
            (act): SiLU(inplace=True)
          )
          (cv2): Conv(
            (conv): Conv2d(64, 32, kernel_size=(1, 1), stride=(1, 1))
            (act): SiLU(inplace=True)
          )
          (cv3): Conv(
            (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
            (act): SiLU(inplace=True)
          )
          (m): Sequential(
            (0): Bottleneck(
              (cv1): Conv(
                (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1))
  

# Gender Classification

In [3]:
# Load gender classification model
gender_model = load_model('cctv_gender_classifier.h5')
gender_classes = ['man', 'woman']

# Initialize DeepSort
tracker = DeepSort(max_age=10, 
                   n_init=3,
                   nms_max_overlap=1.0,
                   max_cosine_distance=0.3,
                   nn_budget=None,
                   override_track_class=None,
                   embedder="mobilenet",
                   half=True,
                   bgr=True,
                   embedder_gpu=False,
                   embedder_model_name=None,
                   embedder_wts=None,
                   polygon=False,
                   today=None)

In [4]:
# Load the action recognition pipeline
action_pipe = pipeline("image-classification", model="rvv-karma/Human-Action-Recognition-VIT-Base-patch16-224", framework="tf")

def preprocess_body(body_crop, target_size=(126, 126)):
    try:
        body_crop = cv2.resize(body_crop, target_size)
        body_crop = body_crop.astype("float32") / 255.0
        body_crop = img_to_array(body_crop)
        body_crop = np.expand_dims(body_crop, axis=0)
        return body_crop
    except Exception as e:
        print(f"Error in preprocess_body: {e}")
        return None


All PyTorch model weights were used when initializing TFViTForImageClassification.

All the weights of TFViTForImageClassification were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFViTForImageClassification for predictions without further training.


In [5]:
# Initialize variables
total_persons = 0
gender_counts = {'man': 0, 'woman': 0}
frame_skip = 2
processing_times = []
fighting_count = 0
lone_woman_flag = False
surrounded_woman_flag = False

# New variables for report generation
sos_events = []
alerts = []
warnings = []
gender_counts_over_time = []

# CCTV Surveillance

In [7]:
# Video paths
video_paths = [
    r"C:\Users\ASUS\Desktop\videos\final vids\crop.mp4",
    r"C:\Users\ASUS\Desktop\videos\final vids\vid epic.mp4",
    r"C:\Users\ASUS\Desktop\videos\final vids\vid front.mp4",
    r"C:\Users\ASUS\Desktop\videos\final vids\vid normal.mp4",
    r"C:\Users\ASUS\Desktop\videos\final vids\vid stalking.mp4",
]

# Open video files
videos = [cv2.VideoCapture(path) for path in video_paths]

cv2.namedWindow("Multi-Video Monitoring", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Multi-Video Monitoring", 1280, 720)

tracker = DeepSort()
CONFIDENCE_THRESHOLD = 0.25
NMS_THRESHOLD = 0.3
frame_skip = 2
frame_count = 0

alerts = []
big_frame_index = 0  # Initially, the first video is in the larger frame

# --- NEW ADDITIONS ---
ALERT_COOLDOWN = 8  # seconds
last_alert_time = None

# Per-camera sensitivity (adjust as needed)
camera_thresholds = {
    0: 0.95,   # Camera 1
    1: 0.75,   # Camera 2
    2: 0.95,   # Camera 3
    3: 0.95,   # Camera 4
    4: 0.75    # Camera 5 (more sensitive)
}

ALERT_COOLDOWN = 8
last_alert_time = None

while True:
    frames = []
    alert_triggered = False
    alert_video_index = None

    for i, video in enumerate(videos):
        ret, frame = video.read()

        if not ret:
            video.set(cv2.CAP_PROP_POS_FRAMES, 0)
            ret, frame = video.read()

        frame = cv2.resize(frame, (320, 240))

        # Detection every frame_skip
        if frame_count % frame_skip == 0:

            results = model(frame)
            detections = results.xyxy[0].cpu().numpy()

            person_count = 0

            for det in detections:
                x1, y1, x2, y2, conf, cls = det
                if conf >= CONFIDENCE_THRESHOLD:
                    person_count += 1
                    cv2.rectangle(frame,
                                  (int(x1), int(y1)),
                                  (int(x2), int(y2)),
                                  (0, 255, 0), 2)

            # ---- Context-Aware Action Logic ----
            if person_count >= 2:

                rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                pil_image = Image.fromarray(rgb_frame)

                action_result = action_pipe(pil_image)

                base_threshold = camera_thresholds.get(i, 0.90)

                # Context Boosting
                current_time = datetime.datetime.now()
                is_night_time = current_time.hour < 6 or current_time.hour >= 18

                for item in action_result:

                    if item['label'] == 'Fighting':

                        threat_score = item['score']

                        # Crowd boost
                        if person_count >= 3:
                            threat_score += 0.04

                        # Night boost
                        if is_night_time:
                            threat_score += 0.04

                        if threat_score > base_threshold:

                            # Cooldown check
                            if (last_alert_time is None or
                                (current_time - last_alert_time).total_seconds() > ALERT_COOLDOWN):

                                alert_triggered = True
                                alert_video_index = i
                                last_alert_time = current_time

                                alert_msg = (
                                    f"FIGHT detected in Camera {i+1} "
                                    f"at {current_time.strftime('%I:%M:%S %p')}"
                                )

                                alerts.append(alert_msg)
                                print("ALERT:", alert_msg)

                                break

        # Camera Label
        cv2.putText(frame, f"Camera {i+1}",
                    (10, 20),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (255, 255, 255),
                    1)

        frames.append(frame)

    # Switch priority display
    if alert_triggered:
        big_frame_index = alert_video_index

    # ---- Display Layout ----
    if len(frames) > 0:

        while len(frames) < 5:
            frames.append(np.zeros((240, 320, 3), dtype=np.uint8))

        big_frame = cv2.resize(frames[big_frame_index], (640, 480))

        small_frames = [frames[j] for j in range(5) if j != big_frame_index]

        top_row = np.hstack(small_frames[:2])
        bottom_row = np.hstack(small_frames[2:])
        side_display = np.vstack((top_row, bottom_row))

        final_display = np.hstack((big_frame, side_display))

        cv2.imshow("Multi-Video Monitoring", final_display)

    frame_count += 1

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


ALERT: FIGHT detected in Camera 2 at 11:23:48 AM


In [8]:
# Cleanup
for video in videos:
    video.release()

cv2.destroyAllWindows()

In [7]:
# Release resources
video.release()
if output_video is not None:
    output_video.release()
cv2.destroyAllWindows()
if sos_recording:
    print(f"SOS video saved as: {output_filename}")

NameError: name 'output_video' is not defined

In [ ]:
yaar i had made that screen wapping based on priority wala code, but now i am unable to find it, i d k where it went can we re code it from this base code
this one isint the base code ill show you the working base code that works on single frame, you need to apply that on multiple videos and based on scoring will highlight the more potential threat screen



In [19]:
video_frames = [
    {
        "frame_number": 1,
        "detections": [
            {"tracking_id": 1, "gender": "male"},
            {"tracking_id": 2, "gender": "female"}
        ],
    },
    {
        "frame_number": 2,
        "detections": [
            {"tracking_id": 1, "gender": "male"},  # Same male as frame 1
            {"tracking_id": 3, "gender": "female"}  # New female
        ],
    },
    {
        "frame_number": 3,
        "detections": [
            {"tracking_id": 2, "gender": "female"},  # Same female as frame 1
            {"tracking_id": 3, "gender": "female"},  # Same female as frame 2
        ],
    },
]

# Initialize sets to store unique IDs
unique_male_ids = set()
unique_female_ids = set()

# Initialize SOS, alerts, and warnings data
sos_events = [{"timestamp": "2024-12-06 12:00:00 PM"}, {"timestamp": "2024-12-06 12:10:00 PM"}]
alerts = []
warnings = [{"timestamp": "2024-12-06 12:20:00 PM"}, {"timestamp": "2024-12-06 12:30:00 PM"}, {"timestamp": "2024-12-06 12:40:00 PM"}]

# Aggregation function
def aggregate_incidents(incident_list, threshold):
    """
    Aggregates incidents within a threshold time interval.
    """
    if not incident_list:
        return 0

    try:
        incident_times = [datetime.datetime.strptime(event['timestamp'], "%Y-%m-%d %I:%M:%S %p") for event in incident_list]
        incident_times.sort()

        aggregated_count = 1
        for i in range(1, len(incident_times)):
            if (incident_times[i] - incident_times[i - 1]).total_seconds() / 60 > threshold:
                aggregated_count += 1

        return aggregated_count
    except KeyError as e:
        print(f"No valid incidents found with '{e.args[0]}'")
        return 0

# Process each frame to track unique individuals
for frame in video_frames:
    for person in frame['detections']:
        tracking_id = person['tracking_id']
        gender = person['gender']
        
        if gender == "female":
            unique_male_ids.add(tracking_id)
        elif gender == "male":
            unique_female_ids.add(tracking_id)

# Aggregate SOS events, alerts, and warnings
aggregated_sos_events = aggregate_incidents(sos_events, threshold=10)
aggregated_alerts = aggregate_incidents(alerts, threshold=10)
aggregated_warnings = aggregate_incidents(warnings, threshold=10)

# Final counts of unique individuals
total_unique_men = len(unique_male_ids)
total_unique_women = len(unique_female_ids)
total_unique_persons = total_unique_men + total_unique_women

# Generate the updated summary
print("======== Video Analysis Summary ========")
print(f"Aggregated SOS events: {aggregated_sos_events}")
print(f"Aggregated warnings: {aggregated_warnings}")
print(f"Total unique persons: {total_unique_persons}")
print(f"Men: {total_unique_men}") 
print(f"Women: {total_unique_women}") 
print("Location: XYZ")
print(f"Gender counts recorded over time: {len(video_frames)} frames")

======== Video Analysis Summary ========
Aggregated SOS events: 1
Aggregated warnings: 1
Total unique persons: 3
Men: 2
Women: 1
Location: XYZ
Gender counts recorded over time: 3 frames


# REPORT GENERATION

In [18]:
import json
import datetime
import requests
from typing import Dict

def generate_security_report(
    total_unique_persons: int,
    total_unique_men: int,
    total_unique_women: int,
    aggregated_sos_events: int,
    aggregated_warnings: int,
    frame_count: int,
    location: str = "XYZ"
) -> str:
    """
    Generate a security report using the Ollama Gemma model based on surveillance data.
    
    Args:
        total_unique_persons: Total number of unique individuals detected
        total_unique_men: Number of unique men detected
        total_unique_women: Number of unique women detected
        aggregated_sos_events: Number of aggregated SOS events
        aggregated_warnings: Number of aggregated warnings
        frame_count: Number of frames analyzed
        location: Location of surveillance
    
    Returns:
        str: Generated security report
    """
    
    # Structure the data for the prompt
    summary_data = {
        "surveillance_summary": {
            "total_frames_analyzed": frame_count,
            "unique_persons": {
                "total": total_unique_persons,
                "men": total_unique_men,
                "women": total_unique_women
            },
            "incidents": {
                "sos_events": aggregated_sos_events,
                "warnings": aggregated_warnings
            },
            "location": location
        }
    }
    
    # Create the prompt for the LLM
    prompt = f"""As a security analysis system, generate a concise but detailed report based on the following surveillance data:
{json.dumps(summary_data, indent=2)}
Generate a professional report covering:
1. Key statistics (total persons detected, gender distribution)
2. Security incidents (SOS events and warnings)
3. Notable patterns or concerns
4. Recommendations based on the findings
Format the report in clear, concise paragraphs suitable for security personnel."""

    try:
        # Prepare the request to Ollama API
        url = 'http://localhost:11434/api/chat'
        payload = {
            "model": "gemma2:2b",
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            "stream": False
        }
        
        # Send request to local Ollama server
        response = requests.post(url, json=payload)
        
        # Check if request was successful
        if response.status_code == 200:
            # Extract the report text from the response
            report = response.json()['message']['content']
            return report
        else:
            raise ConnectionError(f"API request failed with status code {response.status_code}")
    
    except ConnectionError as e:
        print("\nConnection Error: Please ensure Ollama is running locally with these steps:")
        print("1. Install Ollama from: https://ollama.ai/")
        print("2. Run 'ollama serve' in terminal")
        print("3. In another terminal, run 'ollama pull gemma:2b'")
        raise e
    except Exception as e:
        print(f"\nError generating report: {str(e)}")
        raise e

def save_report(report: str, prefix: str = "surveillance_report") -> str:
    """Save the generated report to a file with timestamp."""
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{prefix}_{timestamp}.txt"
    
    with open(filename, "w", encoding="utf-8") as f:
        f.write(report)
    
    return filename

# Example usage
if __name__ == "__main__":
    try:
        # Using dynamic summary data from video analysis
        total_unique_men = len(unique_male_ids)  # From your video analysis data
        total_unique_women = len(unique_female_ids)  # From your video analysis data
        total_unique_persons = total_unique_men + total_unique_women  # From your video analysis data
        aggregated_sos_events = aggregated_sos_events  # From your video analysis data
        aggregated_warnings = aggregated_warnings  # From your video analysis data
        frame_count = len(video_frames)  # Number of frames in the video analysis
        location = "XYZ"  # Update with location from your system, if needed
        
        # Generate the report using dynamic data
        report = generate_security_report(
            total_unique_persons=total_unique_persons,
            total_unique_men=total_unique_men,
            total_unique_women=total_unique_women,
            aggregated_sos_events=aggregated_sos_events,
            aggregated_warnings=aggregated_warnings,
            frame_count=frame_count,
            location=location
        )
        
        # Save and display the report
        filename = save_report(report)
        
        print("\nGenerated Report:")
        print("=" * 50)
        print(report)
        print("\nReport saved to:", filename)
        
    except Exception as e:
        print("Failed to generate report:", str(e))



Error generating report: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/chat (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000002F2AA5B9A30>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))
Failed to generate report: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/chat (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000002F2AA5B9A30>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))
